In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (9,4)

In [3]:
patients = pd.read_csv('../data/raw/patients.csv', parse_dates=['registration_date'])
vitals = pd.read_csv('../data/raw/vital_signs.csv', parse_dates=['timestamp'])
history = pd.read_csv('../data/raw/clinical_history.csv')
labs = pd.read_csv('../data/raw/laboratory_results.csv', parse_dates=['timestamp'])
outcomes = pd.read_csv('../data/raw/sepsis_outcomes.csv', parse_dates=['diagnosis_time'])

tables = {'patients': patients, 'vitals': vitals, 'history': history, 'labs':labs, 'outcomes': outcomes }
for name, df in tables.items():
    print(f'{name:10s} shape={df.shape}')

patients   shape=(5000, 5)
vitals     shape=(99737, 8)
history    shape=(12459, 6)
labs       shape=(20034, 8)
outcomes   shape=(5000, 6)


In [4]:
patients.head()

,patient_id,age,gender,medical_conditions,registration_date
0,1,66,Male,None reported,2024-12-11
1,2,42,Male,None reported,2025-08-24
2,3,74,Female,None reported,2024-07-18
3,4,77,Female,None reported,2024-08-18
4,5,25,Male,COPD,2024-04-09


In [5]:
vitals.describe().T

,count,mean,min,25%,50%,75%,max,std
observation_id,99737.0,49869.0,1.0,24935.0,49869.0,74803.0,99737.0,28791.736236
patient_id,99737.0,2504.455989,1.0,1263.0,2507.0,3759.0,5000.0,1443.230067
timestamp,99737,2025-01-04 18:19:20.888737536,2024-01-01 04:26:00,2024-07-03 22:21:00,2025-01-05 06:16:00,2025-07-12 07:57:00,2025-12-30 04:43:00,NaN
heart_rate,97234.0,84.825789,41.3,74.1,81.3,91.8,149.0,15.773306
temperature,97254.0,36.964523,33.5,36.52,36.85,37.24,40.32,0.806042
oxygen_saturation,97330.0,96.39427,84.1,95.7,97.0,98.0,100.0,2.458295
respiratory_rate,97231.0,17.72458,8.0,14.9,16.8,19.4,37.8,4.230872
blood_pressure,97253.0,116.945143,55.0,108.1,118.5,127.3,166.7,14.981731


In [6]:
print("Sepsis Prevalance:", outcomes['sepsis_event'].mean().round(3))
outcomes["sepsis_event"].value_counts()

Sepsis Prevalance: 0.46


sepsis_event
False    2700
True     2300
Name: count, dtype: int64

## Data Quality Asseessment

### Checking for missing rows

In [7]:
def missing_report(df, name):
    miss = df.isna().mean().mul(100).round(2)
    miss = miss[miss > 0]
    if len(miss):
        print(f'--- {name}---')
        print(miss.to_string())

for name, df in tables.items():
    missing_report(df, name)

--- vitals---
heart_rate           2.51
temperature          2.49
oxygen_saturation    2.41
respiratory_rate     2.51
blood_pressure       2.49
--- labs---
white_cell_count    3.09
crp                 2.83
lactate             3.07
creatinine          2.88
platelet_count      3.00
--- outcomes---
diagnosis_time    54.0


### Checking for duplicate rows

In [8]:
for name, df in tables.items():
    print(name, 'duplicate rows', df.duplicated().sum())

patients duplicate rows 0
vitals duplicate rows 0
history duplicate rows 0
labs duplicate rows 0
outcomes duplicate rows 0


### Checking for negative values in the numerics

In [9]:
for name, df in tables.items():
    numeric = df.select_dtypes(include=['int64', 'float64'])
    neg = (numeric < 0).sum()
    neg = neg[neg > 0]
    print(f"{name}: {neg.to_string() if len(neg) > 0 else 'No negatives'}")

patients: No negatives
vitals: No negatives
history: No negatives
labs: No negatives
outcomes: No negatives


### Checking for referential integrity

In [10]:
valid_ids = set(patients["patient_id"])
for name, df in [('vitals', vitals), ('history', history), ('labs', labs), ('outcomes', outcomes)]:
    orphans = (~df['patient_id'].isin(valid_ids)).sum()
    print(name, 'orphan patient_id', orphans)

vitals orphan patient_id 0
history orphan patient_id 0
labs orphan patient_id 0
outcomes orphan patient_id 0


In [11]:
ranges = {
    'heart_rate': (30, 220), 'temperature': (32, 43), 'oxygen_saturation': (50, 100),
    'respiratory_rate': (5, 60), 'blood_pressure': (40, 220)

}

for cols, (low, high) in ranges.items():
    bad = ((vitals[cols] < low) | (vitals[cols] > high)).sum()
    print(f'{cols}: {bad} out of range')
 

heart_rate: 0 out of range
temperature: 0 out of range
oxygen_saturation: 0 out of range
respiratory_rate: 0 out of range
blood_pressure: 0 out of range


In [12]:
vitals.columns

Index(['observation_id', 'patient_id', 'timestamp', 'heart_rate',
       'temperature', 'oxygen_saturation', 'respiratory_rate',
       'blood_pressure'],
      dtype='object')

In [13]:
labs.columns


Index(['lab_id', 'patient_id', 'timestamp', 'white_cell_count', 'crp',
       'lactate', 'creatinine', 'platelet_count'],
      dtype='object')

In [14]:
vital_cols = ['heart_rate',
       'temperature', 'oxygen_saturation', 'respiratory_rate',
       'blood_pressure'

]

labs_cols = ['white_cell_count', 'crp',
       'lactate', 'creatinine', 'platelet_count'

]

vitals[vital_cols] = vitals.groupby('patient_id')[vital_cols].transform(lambda s: s.ffill())
vitals[vital_cols] = vitals[vital_cols].fillna(vitals[vital_cols].median())

labs[labs_cols] = labs.groupby('patient_id')[labs_cols].transform(lambda s: s.ffill())
labs[labs_cols] = labs[labs_cols].fillna(labs[labs_cols].median())

print('remaining missing values -> vitals:', vitals[vital_cols].isna().sum().sum(), '& labs', labs[labs_cols].isna().sum().sum())


remaining missing values -> vitals: 0 & labs 0


In [15]:
patients.to_csv('../data/processed/patients_clean.csv', index=False)
vitals.to_csv('../data/processed/vital_signs_clean.csv', index=False)
history.to_csv('../data/processed/clinical_history_clean.csv', index=False)
labs.to_csv('../data/processed/laboratory_results_clean.csv', index=False)
outcomes.to_csv('../data/processed/sepsis_outcomes_clean.csv', index=False)

# EDA

In [16]:
# Age distribution
print("\n=== Age Distribution ===")
print(patients['age'].describe())
print(f"\nAge categories:")
print(pd.cut(patients['age'], bins=[0,20,40,60,80,100]).value_counts().sort_index())


=== Age Distribution ===
count    5000.000000
mean       60.030400
std        17.441706
min        18.000000
25%        48.000000
50%        60.000000
75%        72.000000
max        96.000000
Name: age, dtype: float64

Age categories:
age
(0, 20]        73
(20, 40]      601
(40, 60]     1833
(60, 80]     1863
(80, 100]     630
Name: count, dtype: int64


In [17]:
# Gender distribution
print("\n=== Gender Distribution ===")
print(patients['gender'].value_counts())
print(patients['gender'].value_counts(normalize=True).mul(100).round(2))


=== Gender Distribution ===
gender
Female                 2465
Male                   2384
Other/Not specified     151
Name: count, dtype: int64
gender
Female                 49.30
Male                   47.68
Other/Not specified     3.02
Name: proportion, dtype: float64


In [18]:
# Medical conditions
print("\n=== Medical Conditions ===")
if 'medical_conditions' in patients.columns:
    all_conditions = patients['medical_conditions'].str.split(',').explode().str.strip()
    print(all_conditions.value_counts())


=== Medical Conditions ===
medical_conditions
None reported              1215
COPD                        708
Liver Disease               696
Hypertension                680
Immunosuppression           675
Diabetes                    671
Obesity                     671
Coronary Artery Disease     669
Cancer (active)             661
Chronic Kidney Disease      628
Name: count, dtype: int64


In [19]:
# Vital signs statistics
print("\n=== Vital Signs Statistics ===")
vitals_cols = ['heart_rate', 'temperature', 'oxygen_saturation', 'respiratory_rate', 'blood_pressure']
existing_vitals = [col for col in vitals_cols if col in vitals.columns]
if existing_vitals:
    print(vitals[existing_vitals].describe().T)



=== Vital Signs Statistics ===
                     count        mean        std   min     25%     50%  \
heart_rate         99737.0   84.819945  15.756029  41.3   74.10   81.30   
temperature        99737.0   36.963986   0.805312  33.5   36.52   36.85   
oxygen_saturation  99737.0   96.395705   2.456667  84.1   95.70   97.00   
respiratory_rate   99737.0   17.720758   4.226314   8.0   14.90   16.80   
blood_pressure     99737.0  116.957380  14.975330  55.0  108.20  118.50   

                      75%     max  
heart_rate          91.80  149.00  
temperature         37.24   40.32  
oxygen_saturation   98.00  100.00  
respiratory_rate    19.40   37.80  
blood_pressure     127.30  166.70  


In [20]:
# Check for abnormal vital signs
print("\n=== Abnormal Vital Signs ===")
if 'heart_rate' in vitals.columns:
    abnormal_hr = vitals[(vitals['heart_rate'] < 60) | (vitals['heart_rate'] > 100)]
    print(f"Abnormal heart rate (<60 or >100): {len(abnormal_hr)} records")

if 'oxygen_saturation' in vitals.columns:
    low_oxygen = vitals[vitals['oxygen_saturation'] < 92]
    print(f"Low oxygen saturation (<92%): {len(low_oxygen)} records")

if 'respiratory_rate' in vitals.columns:
    abnormal_resp = vitals[(vitals['respiratory_rate'] < 12) | (vitals['respiratory_rate'] > 20)]
    print(f"Abnormal respiratory rate (<12 or >20): {len(abnormal_resp)} records ({len(abnormal_resp)/len(vitals)*100:.2f}%)")

if 'temperature' in vitals.columns:
    abnormal_temp = vitals[(vitals['temperature'] < 36.0) | (vitals['temperature'] > 38.0)]
    print(f"Abnormal temperature (<36.0°C or >38.0°C): {len(abnormal_temp)} records ({len(abnormal_temp)/len(vitals)*100:.2f}%)")

if 'blood_pressure' in vitals.columns:
    abnormal_bp = vitals[(vitals['blood_pressure'] < 90) | (vitals['blood_pressure'] > 140)]
    print(f"Abnormal blood pressure (<90 or >140): {len(abnormal_bp)} records ({len(abnormal_bp)/len(vitals)*100:.2f}%)")


=== Abnormal Vital Signs ===
Abnormal heart rate (<60 or >100): 18974 records
Low oxygen saturation (<92%): 8202 records
Abnormal respiratory rate (<12 or >20): 24870 records (24.94%)
Abnormal temperature (<36.0°C or >38.0°C): 18072 records (18.12%)
Abnormal blood pressure (<90 or >140): 9608 records (9.63%)


### Laboratory Analysis

In [21]:
print("\n=== Laboratory Results Statistics ===")
lab_cols = ['white_cell_count', 'crp', 'lactate', 'creatinine', 'platelet_count']
existing_labs = [col for col in lab_cols if col in labs.columns]
if existing_labs:
    print(labs[existing_labs].describe().T)

# Check for abnormal lab values
print("\n=== Abnormal Lab Values ===")
if 'white_cell_count' in labs.columns:
    abnormal_wbc = labs[
        (labs['white_cell_count'] < 4) | 
        (labs['white_cell_count'] > 11)
    ]
    print(f"Abnormal WBC (<4 or >11): {len(abnormal_wbc)} records")

if 'crp' in labs.columns:
    high_crp = labs[labs['crp'] > 10]
    print(f"High CRP (>10): {len(high_crp)} records")

if 'lactate' in labs.columns:
    high_lactate = labs[labs['lactate'] > 2.0]
    print(f"High Lactate (>2.0 mmol/L): {len(high_lactate)} records ({len(high_lactate)/len(labs)*100:.2f}%)")

if 'creatinine' in labs.columns:
    high_creatinine = labs[labs['creatinine'] > 1.2]
    print(f"High Creatinine (>1.2 mg/dL): {len(high_creatinine)} records ({len(high_creatinine)/len(labs)*100:.2f}%)")

if 'platelet_count' in labs.columns:
    low_platelets = labs[labs['platelet_count'] < 150]
    print(f"Low Platelet Count (<150): {len(low_platelets)} records ({len(low_platelets)/len(labs)*100:.2f}%)")


=== Laboratory Results Statistics ===
                    count        mean        std   min     25%     50%  \
white_cell_count  20034.0    8.648994   3.667016   1.0    6.51    7.90   
crp               20034.0   29.013812  44.901049   0.5    4.20    7.80   
lactate           20034.0    1.557450   1.141713   0.3    0.87    1.11   
creatinine        20034.0    1.060888   0.387769   0.3    0.80    0.98   
platelet_count    20034.0  244.146251  52.573546  17.0  212.00  248.00   

                       75%     max  
white_cell_count    9.6500   27.44  
crp                14.3000  200.10  
lactate             1.5275    6.51  
creatinine          1.2200    2.80  
platelet_count    280.0000  418.00  

=== Abnormal Lab Values ===
Abnormal WBC (<4 or >11): 4320 records
High CRP (>10): 7366 records
High Lactate (>2.0 mmol/L): 4201 records (20.97%)
High Creatinine (>1.2 mg/dL): 5266 records (26.29%)
Low Platelet Count (<150): 988 records (4.93%)


### Sepsis Outcome Analysis

In [22]:
print("\n=== Sepsis Outcomes ===")
if 'sepsis_event' in outcomes.columns:
    print(outcomes['sepsis_event'].value_counts())
    print(f"Sepsis rate: {outcomes['sepsis_event'].mean()*100:.2f}%")

if 'hospitalisation_required' in outcomes.columns:
    print(outcomes['hospitalisation_required'].value_counts())

if 'outcome_status' in outcomes.columns:
    print("\nOutcome Status:")
    print(outcomes['outcome_status'].value_counts())


=== Sepsis Outcomes ===
sepsis_event
False    2700
True     2300
Name: count, dtype: int64
Sepsis rate: 46.00%
hospitalisation_required
True     2654
False    2346
Name: count, dtype: int64

Outcome Status:
outcome_status
Recovered - discharged    3959
Ongoing treatment          554
Deceased                   399
Transferred                 88
Name: count, dtype: int64


### Merging Tables

In [23]:
# Starting with patients
full_data = patients.copy()

In [24]:
# Merge with vital signs (get latest vital signs per patient)
latest_vitals = vitals.sort_values('timestamp').groupby('patient_id').last().reset_index()
full_data = full_data.merge(latest_vitals, on='patient_id', how='left', suffixes=('', '_vitals'))

In [25]:
full_data.head()

,patient_id,age,gender,medical_conditions,registration_date,observation_id,timestamp,heart_rate,temperature,oxygen_saturation,respiratory_rate,blood_pressure
0,1,66,Male,None reported,2024-12-11,21,2024-12-16 04:05:00,121.9,38.48,91.0,27.7,74.6
1,2,42,Male,None reported,2025-08-24,34,2025-08-29 11:47:00,82.3,36.41,97.5,14.9,101.2
2,3,74,Female,None reported,2024-07-18,55,2024-07-21 10:40:00,123.6,38.67,92.5,18.1,97.5
3,4,77,Female,None reported,2024-08-18,72,2024-08-22 17:45:00,107.2,39.70,90.5,26.1,101.6
4,5,25,Male,COPD,2024-04-09,85,2024-04-12 01:52:00,83.1,36.25,98.9,17.1,119.3


In [26]:
# Merge with latest lab results
latest_labs = labs.sort_values('timestamp').groupby('patient_id').last().reset_index()
full_data = full_data.merge(latest_labs, on='patient_id', how='left', suffixes=('', '_lab'))

In [27]:
# Merge clinical history directly 
full_data = full_data.merge(history, on='patient_id', how='left', suffixes=('', '_history'))

In [28]:
# Merge with sepsis outcomes
full_data = full_data.merge(outcomes, on='patient_id', how='left')

print(f"Full dataset shape: {full_data.shape}")
print(f"Columns: {full_data.columns.tolist()}")

Full dataset shape: (12459, 29)
Columns: ['patient_id', 'age', 'gender', 'medical_conditions', 'registration_date', 'observation_id', 'timestamp', 'heart_rate', 'temperature', 'oxygen_saturation', 'respiratory_rate', 'blood_pressure', 'lab_id', 'timestamp_lab', 'white_cell_count', 'crp', 'lactate', 'creatinine', 'platelet_count', 'history_id', 'diagnosis_history', 'infection_history', 'medication_history', 'treatment_history', 'outcome_id', 'sepsis_event', 'diagnosis_time', 'hospitalisation_required', 'outcome_status']


In [29]:
full_data.head()

,patient_id,age,gender,medical_conditions,registration_date,observation_id,timestamp,heart_rate,temperature,oxygen_saturation,respiratory_rate,blood_pressure,lab_id,timestamp_lab,white_cell_count,crp,lactate,creatinine,platelet_count,history_id,diagnosis_history,infection_history,medication_history,treatment_history,outcome_id,sepsis_event,diagnosis_time,hospitalisation_required,outcome_status
0,1,66,Male,None reported,2024-12-11,21,2024-12-16 04:05:00,121.9,38.48,91.0,27.7,74.6,6,2024-12-15 22:11:00,13.47,123.2,3.40,1.65,137.0,1,COPD,Bloodstream,Lisinopril,Dialysis,1,True,2024-12-13 05:36:00,True,Recovered - discharged
1,1,66,Male,None reported,2024-12-11,21,2024-12-16 04:05:00,121.9,38.48,91.0,27.7,74.6,6,2024-12-15 22:11:00,13.47,123.2,3.40,1.65,137.0,2,Diabetes,Bloodstream,Warfarin,Oxygen therapy,1,True,2024-12-13 05:36:00,True,Recovered - discharged
2,1,66,Male,None reported,2024-12-11,21,2024-12-16 04:05:00,121.9,38.48,91.0,27.7,74.6,6,2024-12-15 22:11:00,13.47,123.2,3.40,1.65,137.0,3,Obesity,Urinary Tract,Amlodipine,Physiotherapy,1,True,2024-12-13 05:36:00,True,Recovered - discharged
3,1,66,Male,None reported,2024-12-11,21,2024-12-16 04:05:00,121.9,38.48,91.0,27.7,74.6,6,2024-12-15 22:11:00,13.47,123.2,3.40,1.65,137.0,4,COPD,Respiratory,Prednisone,IV fluids,1,True,2024-12-13 05:36:00,True,Recovered - discharged
4,2,42,Male,None reported,2025-08-24,34,2025-08-29 11:47:00,82.3,36.41,97.5,14.9,101.2,10,2025-08-29 08:25:00,8.12,8.5,0.51,0.81,338.0,5,Chronic Kidney Disease,Respiratory,Lisinopril,Physiotherapy,2,False,NaT,True,Recovered - discharged


In [30]:
# Analyse vital signs trends
if 'timestamp' in vitals.columns:
    print("\n=== Vital Signs Timeline ===")
    
    # Get date range
    print(f"Date range: {vitals['timestamp'].min()} to {vitals['timestamp'].max()}")
    
    # Vital signs per patient over time
    vitals_per_patient = vitals.groupby('patient_id').size()
    print(f"\nAverage vitals records per patient: {vitals_per_patient.mean():.2f}")
    print(f"Max vitals records per patient: {vitals_per_patient.max()}")
    print(f"Min vitals records per patient: {vitals_per_patient.min()}")


=== Vital Signs Timeline ===
Date range: 2024-01-01 04:26:00 to 2025-12-30 04:43:00

Average vitals records per patient: 19.95
Max vitals records per patient: 30
Min vitals records per patient: 10


In [31]:
print("\n=== Correlation Analysis ===")
# Select numeric columns from merged data
numeric_cols = full_data.select_dtypes(include=['int64', 'float64']).columns
if len(numeric_cols) > 0:
    correlation_matrix = full_data[numeric_cols].corr()
    
    # Show correlations with sepsis_event
    if 'sepsis_event' in full_data.columns:
        sepsis_corr = correlation_matrix['sepsis_event'].sort_values(ascending=False)
        print("Top correlations with sepsis event:")
        print(sepsis_corr.head(10))


=== Correlation Analysis ===


KeyError: 'sepsis_event'

In [ ]:
 correlation_matrix.head()

,patient_id,age,observation_id,heart_rate,temperature,oxygen_saturation,respiratory_rate,blood_pressure,lab_id,white_cell_count,crp,lactate,creatinine,platelet_count,outcome_id_x,history_id,outcome_id_y
patient_id,1.000000,-0.022669,0.999872,-0.007739,-0.010908,-0.004470,-0.028563,0.067407,0.999950,-0.036053,-0.034611,0.000383,-0.016242,0.029979,1.000000,0.999905,1.000000
age,-0.022669,1.000000,-0.021559,0.040332,0.016994,-0.038528,0.023693,-0.073004,-0.022234,0.037329,0.024039,0.055186,0.073240,0.055254,-0.022669,-0.022632,-0.022669
observation_id,0.999872,-0.021559,1.000000,-0.007635,-0.012776,-0.004087,-0.029528,0.067952,0.999873,-0.036335,-0.035487,-0.000412,-0.016069,0.029975,0.999872,0.999659,0.999872
heart_rate,-0.007739,0.040332,-0.007635,1.000000,0.333021,-0.572000,0.614374,-0.438781,-0.007590,0.333652,0.684635,0.602452,0.501762,-0.353320,-0.007739,-0.007194,-0.007739
temperature,-0.010908,0.016994,-0.012776,0.333021,1.000000,-0.336901,0.350387,-0.214147,-0.011035,0.192843,0.450061,0.402598,0.316525,-0.196143,-0.010908,-0.010074,-0.010908
